In [5]:
import requests
import json
import os
from dotenv import load_dotenv

load_dotenv()

def post_request_with_body(endpoint, page=1, output_folder="./sources/finca_raiz"):

    request_headers = {
        "Content-Type": "application/json"
    }

    body = {
    "variables": {
        "rows": 100,
        "params": {
            "page": 50,
            "order": 2,
            "bedroomsExactMode": False,
            "bathroomsExactMode": False,
            "operation_type_id": 2,
            "currencyID": 4,
            "m2Currency": 4,
            "locations": [
                {
                    "type": "CITY",
                    "id": "65d441f3-a239-4111-bc5b-01c5a268869f",
                    "name": "Bogotá",
                    "estate": {
                        "name": "Bogotá, d.c.",
                        "id": "2d9f0ad9-8b72-4364-a7dc-e161d7dddb4d",
                        "slug": "state-colombia-11-bogota-dc"
                    }
                }
            ]
        },
        "page": page,
        "source": 10
    },
    "query": ""
}
    
    print(f"Making POST request to: {endpoint}")
    print(f"Page: {page}")
    
    response = requests.post(endpoint, json=body, headers=request_headers)
    response.raise_for_status()
    
    try:
        data = response.json()
        print(f"Response status: {response.status_code}")
        
        if "hits" not in data:
            raise KeyError("'hits' key not found in response")
        
        hits = data["hits"]
        
        os.makedirs(output_folder, exist_ok=True)
        output_file = os.path.join(output_folder, f"finca_raiz_page_{page}.json")
        
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(hits, f, indent=2, ensure_ascii=False)
        
        print(f"Saved {len(hits)} hits to {output_file}")
        return data
        
    except json.JSONDecodeError:
        print(f"Response status: {response.status_code}")
        print(f"Response text: {response.text}")
        return {"status_code": response.status_code, "text": response.text}

In [6]:
def get_all_pages_paginated(endpoint, limit, start_page=1, output_folder="./sources/finca_raiz"):
    all_responses = []
    
    for page in range(start_page, limit + 1):
        print(f"\n=== Processing page {page}/{limit} ===")
        try:
            response = post_request_with_body(endpoint, page=page, output_folder=output_folder)
            
            if "hits" in response and len(response["hits"]) == 0:
                print(f"No hits found on page {page}. Stopping pagination.")
                break
                
        except Exception as e:
            print(f"Error processing page {page}: {e}")
            break
    
    print(f"\n=== Summary ===")
    print(f"Processed {len(all_responses)} pages")


In [4]:
endpoint = "https://search-service.fincaraiz.com.co/api/v1/properties/search"
response = post_request_with_body(endpoint, page=1)

Making POST request to: https://search-service.fincaraiz.com.co/api/v1/properties/search
Page: 1
Response status: 200
Saved 3 hits to ./sources/finca_raiz/finca_raiz_page_1.json


In [7]:
endpoint = "https://search-service.fincaraiz.com.co/api/v1/properties/search"
all_hits = get_all_pages_paginated(endpoint, limit=110, start_page=1)


=== Processing page 1/110 ===
Making POST request to: https://search-service.fincaraiz.com.co/api/v1/properties/search
Page: 1
Response status: 200
Saved 3 hits to ./sources/finca_raiz/finca_raiz_page_1.json

=== Processing page 2/110 ===
Making POST request to: https://search-service.fincaraiz.com.co/api/v1/properties/search
Page: 2
Response status: 200
Saved 3 hits to ./sources/finca_raiz/finca_raiz_page_2.json

=== Processing page 3/110 ===
Making POST request to: https://search-service.fincaraiz.com.co/api/v1/properties/search
Page: 3
Response status: 200
Saved 3 hits to ./sources/finca_raiz/finca_raiz_page_3.json

=== Processing page 4/110 ===
Making POST request to: https://search-service.fincaraiz.com.co/api/v1/properties/search
Page: 4
Response status: 200
Saved 3 hits to ./sources/finca_raiz/finca_raiz_page_4.json

=== Processing page 5/110 ===
Making POST request to: https://search-service.fincaraiz.com.co/api/v1/properties/search
Page: 5
Response status: 200
Saved 3 hits to